# CPRI Hackathon — Task 01: Person 2
## P2-Stage 3: Unsupervised Anomaly Detection

This notebook demonstrates the **independent, unsupervised anomaly detection pipeline** for Task 01.

### Key Methodology:
1. **Unsupervised Fitting**: Isolation Forest and LOF models are fitted **strictly without using `Validity_Label`**.
2. **Target & ID Exclusion**: `Reference_Parameter`, `Test_ID`, and `Validity_Label` are excluded from feature matrices.
3. **Score Standardization**: Standardized so **HIGHER score = MORE anomalous**.
4. **Supervised vs. Unsupervised Agreement**: Reuses Stage 2 OOF predictions to construct 4 agreement groups and extract disagreement records for Person 1.

In [ ]:
import os, sys, pandas as pd, numpy as np
sys.path.insert(0, '../src')
from stage3_features import load_stage3_datasets, prepare_stage3_feature_sets
from stage3_anomaly import IsolationForestAnomalyDetector, LOFAnomalyDetector, evaluate_unsupervised_anomaly

dataset_path = '../data/CPRI_Hackathon_Screening_Dataset_PARTICIPANT.xlsx'
df_tr_flagged, df_te_flagged = load_stage3_datasets(dataset_path)
print(f'Loaded Training_Data: {len(df_tr_flagged)} records, Test_Data: {len(df_te_flagged)} records.')

In [ ]:
# Prepare feature sets & constant feature analysis
prep = prepare_stage3_feature_sets(df_tr_flagged, df_te_flagged)
X_train_dict = prep['X_train']
y_train = prep['y_train']

print('Detected Constant Features (Excluded from X):')
for c_col, c_val in prep['constant_features'].items():
    print(f'  - {c_col}: {c_val}')

In [ ]:
# Execute Anomaly Models across Sets A, B, C, D
results = []
for set_key, f_name in [('Set_A', 'Set A (Raw)'), ('Set_B', 'Set B (Raw+Eng)'), ('Set_C', 'Set C (Raw+Quality)'), ('Set_D', 'Set D (Raw+Eng+Quality)')]:
    X_tr = X_train_dict[set_key]
    
    # Isolation Forest
    iforest = IsolationForestAnomalyDetector(n_estimators=200, contamination='auto', random_state=42).fit(X_tr)
    results.append(evaluate_unsupervised_anomaly(iforest.score_samples(X_tr), iforest.predict(X_tr), y_train, 'Isolation Forest', f_name))
    
    # LOF
    lof = LOFAnomalyDetector(n_neighbors=20, contamination='auto').fit(X_tr)
    results.append(evaluate_unsupervised_anomaly(lof.score_samples(X_tr), lof.predict(X_tr), y_train, 'LOF', f_name))

pd.DataFrame(results)[['model_name', 'feature_set', 'num_flagged', 'pct_flagged', 'roc_auc', 'pr_auc', 'invalid_precision', 'invalid_recall', 'invalid_f1']]